# XJTU reference-test SOH transitions and RL feasibility

Assess whether the effect of variable discharge can be forecast on held-out batteries, using standardized reference performance tests (RPTs) as the SOH label. Random Walk is the primary cohort. R3 is a separate comparison with a recurring discharge sequence and different charge protocol.

Run all cells from the LifeGuard-RL project. The notebook reads the 16 existing BatteryLife XJTU cell pickles sequentially from the extracted folder or the local ZIP. It runs on one CPU thread.

In [1]:
import os
for name in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS",
             "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS"):
    os.environ[name] = "1"

import json
import pickle
from pathlib import Path, PurePosixPath
from zipfile import ZIP_DEFLATED, ZipFile

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from threadpoolctl import threadpool_limits

DATASET_PATH = None  # Extracted XJTU directory or BatteryLife's XJTU.zip.
OUTPUT_ROOT = None
OUTPUT_NAME = "lifeguard_xjtu_discharge_soh_results"
COHORTS = {
    "RW": dict(name="Random Walk", interval_cycles=10, diagnostic_charge_a=2, operating_charge_a=6,
               currents_a=(2, 3, 4, 5, 6, 7, 8), duration_grid_s=(120, 180, 240, 300, 360)),
    "R3": dict(name="R3", interval_cycles=5, diagnostic_charge_a=4, operating_charge_a=4,
               currents_a=(1, 2, 4, 6, 10)),
}
HORIZONS = (1, 3)
ALPHAS = (1, 10, 100)
BOOTSTRAP_DRAWS = 2000
SEED = 42
HEALTH_BANDS = ("90% and above", "80% to under 90%", "Below 80%")
MIN_ACTION_BATTERIES = 4  # Minimum per action and band within each training fold.
MIN_COMMON_ACTIONS = 2
MODEL_LABELS = {
    "persistence": "Persistence",
    "trend": "Past trend",
    "no_action": "No-action Ridge",
    "current_only": "Current-only Ridge",
    "action": "Action-aware Ridge",
    "realized": "Realized-exposure Ridge (diagnostic)",
}

cwd = Path.cwd()
project = next((p for p in (cwd, *cwd.parents) if (p / "README_LifeGuard_RL.md").is_file()), None)
if project is None:
    raise FileNotFoundError("Open this notebook from within the LifeGuard-RL project.")
if DATASET_PATH is not None:
    source = Path(DATASET_PATH).expanduser()
else:
    extracted = project / "Dataset" / "XJTU"
    source = extracted if extracted.is_dir() else extracted.with_suffix(".zip")
output = (Path(OUTPUT_ROOT).expanduser() if OUTPUT_ROOT else project / "reports") / OUTPUT_NAME

## What counts as a health measurement

Use only cycles marked RPT that match the expected diagnostic charge current, 1C discharge and approximately 2.5 V cutoff. Check the recorded capacity counter against integrated discharge current. SOH is 100 × diagnostic discharge Ah ÷ nominal 2.0 Ah. Increases in measured diagnostic capacity are preserved.

Operating cycles instead discharge near 3.0 V. For Random Walk, reconstruct the initial current and uninterrupted duration when the first plateau has a verified transition to a different current before cutoff and its duration lies on the documented two- to six-minute grid. The forecast receives these first plateaus as a prescribed schedule. The logged final pulse can end at the voltage cutoff, and its requested duration is ambiguous. The retrospective experiment records realized time, throughput and current exposure separately.

In [2]:
class XJTUUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        raise pickle.UnpicklingError(f"Unexpected global in XJTU data: {module}.{name}")


def requested_cells():
    return [(cohort, f"XJTU_{cohort}_battery-{i}") for cohort in COHORTS for i in range(1, 9)]


def current_mode(current, time, grid):
    initial = current[(time - time[0] <= 10)]
    magnitude = float(np.median(initial))
    return next((level for level in grid if abs(level - magnitude) <= 0.12
                 and len(initial) >= 3 and np.mean(abs(initial - level) <= 0.12) >= 0.8), None)


def audit_cycle(cycle, nominal, setup):
    idx = int(cycle["cycle_number"])
    kind = cycle.get("attribute", "")
    row = dict(cycle=idx, kind=kind, description=cycle.get("step_description", ""),
               status="invalid_trace", diagnostic_ah=np.nan, diagnostic_soh_pct=np.nan,
               first_current_a=np.nan, first_plateau_seconds=np.nan, verified_first_duration_s=np.nan,
               discharge_minutes=np.nan, discharge_wh=np.nan, discharge_amp2_hours=np.nan,
               negative_end_voltage=np.nan, charge_cc_a=np.nan, capacity_difference_ah=np.nan)
    time, voltage, current, capacity = (np.asarray(cycle[field], dtype=float) for field in
                                        ("time_in_s", "voltage_in_V", "current_in_A", "discharge_capacity_in_Ah"))
    if (len(time) < 3 or any(x.ndim != 1 or len(x) != len(time) or not np.isfinite(x).all()
                           for x in (time, voltage, current, capacity))):
        return row
    dt = np.diff(time)
    negative = current < -0.25
    intervals = negative[:-1] & negative[1:]
    if np.any(dt < 0) or not intervals.any():
        return row
    active = np.flatnonzero(negative)
    start, end = active[0], active[-1]
    discharge = -current[active]
    dtime = time[active]
    positive = current[:start][current[:start] > 0.25]
    if not len(positive):
        return row
    cc = float(np.quantile(positive, 0.9))
    delivered_ah = float(np.dot(dt[intervals], (-current[:-1] - current[1:])[intervals]) / 7200)
    power = -current * voltage
    wh = float(np.dot(dt[intervals], (power[:-1] + power[1:])[intervals]) / 7200)
    amp2 = float(np.dot(dt[intervals], (current[:-1]**2 + current[1:]**2)[intervals]) / 7200)
    minutes = float(dt[intervals].sum() / 60)
    meter = float(np.max(capacity[active]))
    terminal_v = float(voltage[end])
    row.update(discharge_minutes=minutes, discharge_wh=wh, discharge_amp2_hours=amp2,
               negative_end_voltage=terminal_v, charge_cc_a=cc, capacity_difference_ah=meter-delivered_ah)
    if kind == "RPT":
        expected_charge = setup["diagnostic_charge_a"]
        if (abs(cc - expected_charge) > 0.08 * expected_charge
                or np.mean(abs(discharge - nominal) <= 0.08 * nominal) < 0.98
                or abs(terminal_v - 2.5) > 0.08):
            row["status"] = "different_rpt_protocol"
        elif (meter <= 0 or abs(delivered_ah - meter) > max(0.020, 0.015 * meter)):
            row["status"] = "rpt_meter_disagreement"
        else:
            row.update(status="valid_rpt", diagnostic_ah=meter, diagnostic_soh_pct=100 * meter / nominal)
        return row
    if kind != "Cycling":
        row["status"] = "unknown_cycle_type"
        return row

    expected_charge = setup["operating_charge_a"]
    first = current_mode(discharge, dtime, setup["currents_a"])
    if first is None:
        row["status"] = "unverified_initial_current"
        return row
    aligned = abs(discharge - first) <= .12
    plateau_start = int(np.flatnonzero(aligned)[0])
    ends = np.flatnonzero((~aligned | np.r_[False, np.diff(active) != 1])[plateau_start + 1:])
    plateau_end = plateau_start + int(ends[0]) if len(ends) else len(active) - 1
    first_seconds = float(dtime[plateau_end] - dtime[plateau_start])
    # Restrict the schedule feature to a completed plateau with a verified
    # change to a different current before the cutoff.
    grid = setup.get("duration_grid_s", ())
    duration = np.nan
    next_index = plateau_end + 1
    if (grid and next_index < len(active) and active[next_index] == active[plateau_end] + 1
            and not aligned[next_index] and voltage[active[plateau_end]] > 3.03):
        duration = next((item for item in grid if abs(item - first_seconds) <= 2), np.nan)
    row.update(first_current_a=first, first_plateau_seconds=first_seconds, verified_first_duration_s=duration)
    if abs(cc - expected_charge) > 0.08 * expected_charge:
        row["status"] = "different_operating_charge"
    elif abs(terminal_v - 3.0) > 0.09 or minutes <= 0 or wh <= 0:
        row["status"] = "unverified_operating_cutoff"
    elif setup["interval_cycles"] == 5 and np.mean(abs(discharge - first) <= .12) < .98:
        row["status"] = "unexpected_r3_current_switch"
    else:
        row["status"] = "valid_operating"
    return row


def read_battery(stream, cohort, cell):
    battery = XJTUUnpickler(stream).load()
    nominal = float(battery["nominal_capacity_in_Ah"])
    if battery["cell_id"] != cell or not np.isclose(nominal, 2.0):
        raise ValueError(f"Unexpected identity or nominal Ah for {cell}.")
    audit = [audit_cycle(cycle, nominal, COHORTS[cohort]) for cycle in battery["cycle_data"]]
    steps = [item["cycle"] for item in audit]
    if steps != sorted(set(steps)):
        raise ValueError(f"Battery {cell} has repeated or unordered cycle numbers.")
    return [dict(cohort=cohort, cell_id=cell, **item) for item in audit]


def audit_inputs(path):
    results = []
    if path.is_dir():
        for cohort, cell in requested_cells():
            with (path / f"{cell}.pkl").open("rb") as stream:
                results.extend(read_battery(stream, cohort, cell))
            print(f"Audited {cell}", flush=True)
        return pd.DataFrame(results)
    with ZipFile(path) as archive:
        members = {}
        for name in archive.namelist():
            entry = PurePosixPath(name)
            if entry.suffix == ".pkl" and not any(part.startswith(".") for part in entry.parts):
                if entry.stem in members:
                    raise ValueError(f"Duplicated cell file in archive: {entry.stem}.")
                members[entry.stem] = name
        for cohort, cell in requested_cells():
            with archive.open(members[cell]) as stream:
                results.extend(read_battery(stream, cohort, cell))
            print(f"Audited {cell}", flush=True)
    return pd.DataFrame(results)


with threadpool_limits(limits=1):
    cycles = audit_inputs(source)
print(cycles.groupby(["cohort", "status"]).size().to_string())

Audited XJTU_RW_battery-1
Audited XJTU_RW_battery-2
Audited XJTU_RW_battery-3
Audited XJTU_RW_battery-4
Audited XJTU_RW_battery-5
Audited XJTU_RW_battery-6
Audited XJTU_RW_battery-7
Audited XJTU_RW_battery-8
Audited XJTU_R3_battery-1
Audited XJTU_R3_battery-2
Audited XJTU_R3_battery-3
Audited XJTU_R3_battery-4
Audited XJTU_R3_battery-5
Audited XJTU_R3_battery-6
Audited XJTU_R3_battery-7
Audited XJTU_R3_battery-8
cohort  status                    
R3      valid_operating               4725
        valid_rpt                      945
RW      different_operating_charge     160
        valid_operating               1650
        valid_rpt                      173


In [3]:
def assemble_intervals(cycles):
    records = []
    for (cohort, cell), group in cycles.groupby(["cohort", "cell_id"], sort=True):
        trace = group.set_index("cycle").sort_index()
        setup = COHORTS[cohort]
        diagnostics = trace.loc[trace.kind.eq("RPT")]
        operating_age = trace.kind.eq("Cycling").cumsum()
        for start, end in zip(diagnostics.index[:-1], diagnostics.index[1:]):
            following = trace.loc[start + 1:end - 1]
            before, after = diagnostics.loc[start], diagnostics.loc[end]
            status = "valid"
            if before.status != "valid_rpt" or after.status != "valid_rpt":
                status = "failed_reference_check"
            elif (len(following) != setup["interval_cycles"] or end - start - 1 != len(following)
                  or not following.kind.eq("Cycling").all()):
                status = "unexpected_interval_length"
            elif not following.status.eq("valid_operating").all():
                status = "failed_operating_check"
            elif cohort == "RW" and following.verified_first_duration_s.isna().any():
                status = "unverified_first_duration"
            elif cohort == "R3" and sorted(following.first_current_a) != list(setup["currents_a"]):
                status = "unexpected_r3_rate_mix"

            action = following.first_current_a
            scheduled_minutes = following.verified_first_duration_s / 60
            records.append(dict(cohort=cohort, cell_id=cell, start_cycle=int(start), end_cycle=int(end),
                                operating_age=int(operating_age.loc[start]), operating_cycles=len(following), status=status,
                                soh_before=before.diagnostic_soh_pct, soh_after=after.diagnostic_soh_pct,
                                nominal_ah=2.0, first_current_mean_a=action.mean(),
                                initial_current_square_mean_a2=(action**2).mean(),
                                initial_duration_mean_min=scheduled_minutes.mean() if cohort == "RW" else 0,
                                initial_current_square_minutes=(action**2 * scheduled_minutes).mean() if cohort == "RW" else 0,
                                realized_discharge_minutes=following.discharge_minutes.sum(min_count=len(following)),
                                realized_amp2_hours=following.discharge_amp2_hours.sum(min_count=len(following)),
                                realized_wh=following.discharge_wh.sum(min_count=len(following))))
    return pd.DataFrame(records)


def add_histories(intervals):
    data = intervals.loc[intervals.status.eq("valid")].copy().reset_index(drop=True)
    end_at = {(row.cell_id, row.end_cycle): row for row in data.itertuples(index=False)}
    histories = []
    for row in data.itertuples(index=False):
        previous = end_at.get((row.cell_id, row.start_cycle))
        earlier = end_at.get((row.cell_id, previous.start_cycle)) if previous is not None else None
        histories.append((previous.soh_before if previous is not None else np.nan,
                          earlier.soh_before if earlier is not None else np.nan))
    data[["previous_soh", "earlier_soh"]] = histories
    return data


def chain_indices(data, horizon):
    by_start = {(row.cell_id, row.start_cycle): row.Index for row in data.itertuples()}
    routes = []
    for row in data.itertuples():
        path = [row.Index]
        for _ in range(1, horizon):
            forward = by_start.get((row.cell_id, data.at[path[-1], "end_cycle"]))
            if forward is None:
                break
            path.append(forward)
        if len(path) == horizon:
            routes.append(path)
    return np.asarray(routes, dtype=int).reshape(-1, horizon)


def health_band(value):
    if value >= 90:
        return HEALTH_BANDS[0]
    if value >= 80:
        return HEALTH_BANDS[1]
    return HEALTH_BANDS[2]


def first_current_support(cycles):
    observations = []
    for cell, trace in cycles.loc[cycles.cohort.eq("RW")].groupby("cell_id"):
        before = None
        for row in trace.sort_values("cycle").itertuples(index=False):
            if row.kind == "RPT":
                before = row if row.status == "valid_rpt" else None
            if (before is not None and row.status == "valid_operating"
                    and 1 <= row.cycle - before.cycle <= COHORTS["RW"]["interval_cycles"]):
                observations.append((cell, row.cycle, before.cycle, before.diagnostic_soh_pct,
                                     health_band(before.diagnostic_soh_pct), int(row.first_current_a),
                                     row.verified_first_duration_s))
    return pd.DataFrame(observations, columns=["cell_id", "cycle", "reference_cycle", "reference_soh_pct",
                                               "health_band", "first_current_a", "first_duration_s"])


def summarize_support(observations, cells):
    records = []
    for excluded in cells:
        training = observations.loc[observations.cell_id.ne(excluded)]
        for band in HEALTH_BANDS:
            group = training.loc[training.health_band.eq(band)]
            for amps in COHORTS["RW"]["currents_a"]:
                for seconds in COHORTS["RW"]["duration_grid_s"]:
                    selected = group.loc[group.first_current_a.eq(amps) & group.first_duration_s.eq(seconds)]
                    records.append((excluded, band, amps, seconds, len(selected), selected.cell_id.nunique()))
    counts = pd.DataFrame(records, columns=["held_out", "health_band", "action_a", "duration_s",
                                           "commands", "training_batteries"])
    relevant = counts.loc[counts.health_band.isin(HEALTH_BANDS[:2])]
    both_bands = (relevant.pivot_table(index=["held_out", "action_a", "duration_s"], columns="health_band",
                                      values="training_batteries", aggfunc="first")
                  .reindex(columns=HEALTH_BANDS[:2]).ge(MIN_ACTION_BATTERIES).all(axis=1))
    common = both_bands.groupby(level=["action_a", "duration_s"]).all()
    return counts, [(int(amps), int(seconds)) for amps, seconds in common.index[common]]


intervals = assemble_intervals(cycles)
data = add_histories(intervals)
raw_checks = (cycles.groupby(["cohort", "cell_id"], as_index=False)
              .agg(records=("cycle", "size"), rpt_marked=("kind", lambda types: types.eq("RPT").sum()),
                   rpt_verified=("status", lambda status: status.eq("valid_rpt").sum()),
                   operating_verified=("status", lambda status: status.eq("valid_operating").sum())))
observed_commands = first_current_support(cycles)
rw_cells = raw_checks.loc[raw_checks.cohort.eq("RW"), "cell_id"]
support, common_actions = summarize_support(observed_commands, rw_cells)
print(intervals.groupby(["cohort", "status"]).size().to_string())
print("RW current–duration pairs meeting both health-band requirements in every training fold:", common_actions)

cohort  status
R3      valid     937
RW      valid     165
RW current–duration pairs meeting both health-band requirements in every training fold: [(2, 120), (2, 180), (2, 240), (2, 300), (2, 360), (3, 120), (3, 180), (3, 240), (3, 300), (3, 360), (4, 120), (4, 180), (4, 240), (4, 300), (4, 360)]


## Held-out forecast comparison

Fit within each cohort independently with one battery held out at a time. The deployable Ridge models predict the SOH change to the next RPT. All receive starting SOH, operating age, and changes over the last two RPT intervals. The current-only version adds the mean current and mean squared current of the prescribed first plateaus. The action-aware version also adds the planned mean duration and the mean of squared current multiplied by planned duration. Later and cutoff-dependent times are outside these features. R3 uses fixed cutoffs and supplies current only.

The retrospective diagnostic model adds the measured duration and squared-current integral for the interval after it occurred. It is a sensitivity check. It is excluded from the RL gate. For all models, three-step forecasts feed predicted SOH back in and use the recorded actions as the prescribed schedule. Historical features stop at the starting RPT. The Ridge penalties are selected on inner held-out training batteries using three-step MAE.

In [4]:
FEATURE_GROUPS = ("no_action", "current_only", "action", "realized")


def matrix(state, on_rows, model):
    soh, prior, earlier = state.T
    known = np.isfinite(prior)
    recent = np.where(known, soh - prior, 0)
    drift = np.where(np.isfinite(earlier), (soh - earlier) / 2, recent)
    base = np.column_stack((soh, on_rows.operating_age.to_numpy(), recent, drift, known))
    if model == "no_action":
        return base
    effects = on_rows[["first_current_mean_a", "initial_current_square_mean_a2"]].to_numpy()
    if model == "current_only":
        return np.column_stack((base, effects))
    timing = on_rows[["initial_duration_mean_min", "initial_current_square_minutes"]].to_numpy()
    if model == "action":
        return np.column_stack((base, effects, timing))
    measured = on_rows[["realized_discharge_minutes", "realized_amp2_hours"]].to_numpy()
    return np.column_stack((base, effects, timing, measured))


def initial_state(rows):
    return rows[["soh_before", "previous_soh", "earlier_soh"]].to_numpy(dtype=float)


def fit_ridge(data, train_mask, model, alpha):
    train = data.loc[train_mask]
    if train.empty:
        raise ValueError("A forecast model received an empty training set.")
    inputs = matrix(initial_state(train), train, model)
    if not np.isfinite(inputs).all():
        raise ValueError("A scored transition has nonfinite predictors.")
    weights = 1 / train.groupby("cell_id").start_cycle.transform("size").to_numpy(dtype=float)
    weights /= weights.mean()
    variable = np.ptp(inputs, axis=0) > 0
    inputs = inputs[:, variable]
    center = np.average(inputs, axis=0, weights=weights)
    residual = inputs - center
    scale = np.sqrt(np.average(residual**2, axis=0, weights=weights))
    learner = Ridge(alpha=alpha).fit(residual / scale,
                                      (train.soh_after - train.soh_before).to_numpy(), sample_weight=weights)

    def predict_change(state, rows):
        values = matrix(state, rows, model)[:, variable]
        return learner.predict((values - center) / scale)
    return predict_change


def rollout(data, paths, model, predict_change=None):
    start = initial_state(data.iloc[paths[:, 0]])
    if model == "persistence":
        return start[:, 0].copy()
    if model == "trend":
        soh, prior, earlier = start.T
        drift = np.where(np.isfinite(earlier), (soh - earlier) / 2,
                         np.where(np.isfinite(prior), soh - prior, 0))
        return soh + drift * paths.shape[1]
    state = start
    for indices in paths.T:
        latest = state[:, 0] + predict_change(state, data.iloc[indices])
        state = np.column_stack((latest, state[:, :2]))
    return state[:, 0]


def select_alpha(data, held_out, paths, model):
    by_battery = data.cell_id.to_numpy()
    loss = {alpha: [] for alpha in ALPHAS}
    for inner in sorted(set(by_battery) - {held_out}):
        scored = paths[by_battery[paths[:, 0]] == inner]
        if not len(scored):
            continue
        training = (by_battery != held_out) & (by_battery != inner)
        truth = data.soh_after.to_numpy()[scored[:, -1]]
        for alpha in ALPHAS:
            fit = fit_ridge(data, training, model, alpha)
            loss[alpha].append(float(np.mean(np.abs(rollout(data, scored, model, fit) - truth))))
    if not any(loss.values()):
        raise ValueError("No training-only three-test windows for model selection.")
    average = {alpha: float(np.mean(values)) for alpha, values in loss.items()}
    best = min(ALPHAS, key=lambda alpha: (average[alpha], -alpha))
    return best, average


def held_out_results(data):
    predictions, selections = [], []
    for cohort, grouped in data.groupby("cohort", sort=True):
        local = grouped.reset_index(drop=True)
        by_battery = local.cell_id.to_numpy()
        paths = {h: chain_indices(local, h) for h in HORIZONS}
        for held_out in sorted(set(by_battery)):
            models = {"persistence": None, "trend": None}
            for model in FEATURE_GROUPS:
                selected, inner_scores = select_alpha(local, held_out, paths[3], model)
                models[model] = fit_ridge(local, by_battery != held_out, model, selected)
                for alpha, score in inner_scores.items():
                    selections.append(dict(cohort=cohort, cell_id=held_out, model=model, alpha=alpha,
                                           inner_mae_3_tests=score, selected=alpha == selected))
            for h, all_paths in paths.items():
                selected_paths = all_paths[by_battery[all_paths[:, 0]] == held_out]
                if not len(selected_paths):
                    continue
                origin = local.iloc[selected_paths[:, 0]]
                end = local.iloc[selected_paths[:, -1]]
                rows = pd.DataFrame(dict(cohort=cohort, cell_id=held_out, horizon_tests=h,
                                         start_cycle=origin.start_cycle.to_numpy(),
                                         end_cycle=end.end_cycle.to_numpy(),
                                         start_soh_pct=origin.soh_before.to_numpy(),
                                         observed_soh_pct=end.soh_after.to_numpy(),
                                         nominal_ah=origin.nominal_ah.to_numpy()))
                for model, predictor in models.items():
                    rows[model] = rollout(local, selected_paths, model, predictor)
                predictions.append(rows)
    return pd.concat(predictions, ignore_index=True), pd.DataFrame(selections)


def verify_forecast_logic(data):
    subset = data.loc[data.cohort.eq("RW")].reset_index(drop=True)
    route = chain_indices(subset, 3)[:1]
    assert len(route) == 1
    start = float(subset.iloc[route[0, 0]].soh_before)
    # A synthetic one-point increase verifies predicted-state feedback at every step.
    change = lambda state, _: state[:, 0] - start + 1
    expected = np.array([start + 7])
    np.testing.assert_allclose(rollout(subset, route, "action", change), expected)
    replacement = subset.copy()
    replacement.loc[route.ravel(), "soh_after"] = -999
    np.testing.assert_array_equal(rollout(subset, route, "action", change),
                                  rollout(replacement, route, "action", change))
    # No-action predictions cannot depend on future measured duration or action fields.
    altered = subset.iloc[route.ravel()].copy()
    state = initial_state(altered)
    unchanged = matrix(state, altered, "no_action")
    altered["first_current_mean_a"] = 99
    altered["realized_discharge_minutes"] = 99
    np.testing.assert_array_equal(matrix(state, altered, "no_action"), unchanged)
    unchanged_time = matrix(state, subset.iloc[route.ravel()], "action")
    altered["first_current_mean_a"] = subset.iloc[route.ravel()].first_current_mean_a
    np.testing.assert_array_equal(matrix(state, altered, "action"), unchanged_time)


verify_forecast_logic(data)
with threadpool_limits(limits=1):
    predictions, model_selection = held_out_results(data)
print(predictions.groupby(["cohort", "horizon_tests"]).agg(windows=("cell_id", "size"),
                                                             batteries=("cell_id", "nunique")).to_string())

                      windows  batteries
cohort horizon_tests                    
R3     1                  937          8
       3                  921          8
RW     1                  165          8
       3                  149          8


## Decision rule for PPO

The Random Walk action-aware model must improve three-test battery-weighted MAE over both the past-trend and no-action models. Both paired battery-bootstrap intervals for the MAE difference must have upper endpoints below zero. Action overlap requires at least two common first-current and duration pairs, each recorded on at least four training batteries in both the ≥90% and 80%–90% reference SOH bands in every leave-one-battery-out split. R3 and the retrospective predictor do not qualify the Random Walk transition.

In [5]:
def confidence(values):
    rng = np.random.default_rng(SEED)
    indexes = rng.integers(0, len(values), size=(BOOTSTRAP_DRAWS, len(values)))
    return np.quantile(values[indexes].mean(axis=1), [0.025, 0.975])


def score(predictions):
    metrics, comparisons = [], []
    for (cohort, horizon), rows in predictions.groupby(["cohort", "horizon_tests"], sort=True):
        errors = {name: rows[name] - rows.observed_soh_pct for name in MODEL_LABELS}
        absolute = pd.DataFrame({model: values.abs() for model, values in errors.items()})
        cell_means = absolute.groupby(rows.cell_id).mean()
        for model, values in errors.items():
            worst = rows.loc[absolute[model].idxmax()]
            n_tail = int(np.ceil(0.10 * len(rows)))
            metrics.append(dict(cohort=cohort, horizon_tests=horizon, model=model, windows=len(rows),
                                batteries=len(cell_means), mae_pp=float(cell_means[model].mean()),
                                worst_decile_mae_pp=float(absolute[model].nlargest(n_tail).mean()),
                                max_error_pp=float(absolute[model].max()),
                                bias_pp=float(values.groupby(rows.cell_id).mean().mean()),
                                worst_cell=worst.cell_id, worst_start_cycle=int(worst.start_cycle),
                                worst_end_cycle=int(worst.end_cycle)))
        for baseline in ("trend", "no_action"):
            paired = (cell_means.action - cell_means[baseline]).to_numpy()
            low, high = confidence(paired)
            comparisons.append(dict(cohort=cohort, horizon_tests=horizon, baseline=baseline,
                                    batteries=len(paired), mean_mae_change_pp=paired.mean(),
                                    ci95_low_pp=low, ci95_high_pp=high, action_better_batteries=int((paired < 0).sum())))
    return pd.DataFrame(metrics), pd.DataFrame(comparisons)


def assess_rl(comparisons, common_actions):
    selected = comparisons.loc[comparisons.cohort.eq("RW") & comparisons.horizon_tests.eq(3)].set_index("baseline")
    vs_trend = bool(selected.at["trend", "ci95_high_pp"] < 0)
    vs_no_action = bool(selected.at["no_action", "ci95_high_pp"] < 0)
    action_overlap = len({amps for amps, _ in common_actions}) >= MIN_COMMON_ACTIONS
    return pd.DataFrame([dict(action_beats_trend=vs_trend, action_beats_no_action=vs_no_action,
                              comparable_action_coverage=action_overlap, common_actions=len(common_actions),
                              advance_to_ppo=vs_trend and vs_no_action and action_overlap)])


with threadpool_limits(limits=1):
    metrics, paired = score(predictions)
    gate = assess_rl(paired, common_actions)
print(metrics[["cohort", "horizon_tests", "model", "mae_pp", "max_error_pp"]].round(4).to_string(index=False))
print(gate.to_string(index=False))

cohort  horizon_tests        model  mae_pp  max_error_pp
    R3              1  persistence  0.2271        1.4000
    R3              1        trend  0.1150        1.2500
    R3              1    no_action  0.0859        0.4537
    R3              1 current_only  0.0859        0.4537
    R3              1       action  0.0859        0.4537
    R3              1     realized  0.0857        0.4530
    R3              3  persistence  0.5776        3.8000
    R3              3        trend  0.2690        2.9000
    R3              3    no_action  0.1926        1.2207
    R3              3 current_only  0.1926        1.2207
    R3              3       action  0.1926        1.2207
    R3              3     realized  0.1910        1.1877
    RW              1  persistence  1.1413        5.8000
    RW              1        trend  0.5919        2.3000
    RW              1    no_action  0.1953        0.7072
    RW              1 current_only  0.1974        0.7171
    RW              1       act

In [6]:
def build_summary(metrics, paired, gate, cycles, intervals, support, common_actions):
    result = gate.iloc[0]
    conclusion = ("The Random Walk evidence meets the prespecified criteria for a PPO pilot."
                  if result.advance_to_ppo
                  else "The Random Walk evidence falls short of the criteria for a PPO pilot.")
    diag = cycles.loc[cycles.kind.eq("RPT")]
    verified = diag.loc[diag.status.eq("valid_rpt")]
    valid = intervals.loc[intervals.status.eq("valid")]
    lines = [
        "# XJTU reference-test SOH forecasting and RL feasibility", "",
        "## Conclusion", "",
        f"**{conclusion}** This experiment predicts SOH measured during standardized capacity tests "
        "from the preceding health measurements and a specified discharge schedule.", "",
        "## Reference test audit", "",
        "| Cohort | Batteries | Reference tests verified | Accepted intervals | Intervals with measured SOH increase |",
        "| --- | ---: | ---: | ---: | ---: |",
    ]
    for group, spec in COHORTS.items():
        expected = diag.loc[diag.cohort.eq(group)]
        checked = verified.loc[verified.cohort.eq(group)]
        complete = valid.loc[valid.cohort.eq(group)]
        lines.append(f"| {spec['name']} | {expected.cell_id.nunique()} | {len(checked)}/{len(expected)} | "
                     f"{len(complete)} | {int(complete.soh_after.gt(complete.soh_before).sum())} |")
    lines.extend([
        "",
        "SOH is diagnostic discharge capacity divided by nominal 2.0 Ah. The audit verifies the discharge current, "
        "voltage cutoff, charge current and agreement between measured and integrated capacity. Random Walk "
        "intervals have ten operating cycles and R3 intervals have five. Changes in measured SOH can include "
        "short-term recovery and measurement variability.", "",
        "## Held-out forecast errors", "",
        "All errors are SOH percentage points. MAE gives each battery equal weight. The tail uses the largest "
        "10% of absolute errors across scored windows. The realized-exposure row uses data recorded during the "
        "forecast interval and is a retrospective diagnostic.", "",
        "| Cohort | Tests ahead | Model | Windows | Mean absolute error | Worst 10% mean | Maximum error |",
        "| --- | ---: | --- | ---: | ---: | ---: | ---: |",
    ])
    for group, spec in COHORTS.items():
        for horizon in HORIZONS:
            scores = metrics.loc[metrics.cohort.eq(group) & metrics.horizon_tests.eq(horizon)].set_index("model")
            for model, label in MODEL_LABELS.items():
                row = scores.loc[model]
                lines.append(f"| {spec['name']} | {horizon} | {label} | {int(row.windows)} | {row.mae_pp:.3f} | "
                             f"{row.worst_decile_mae_pp:.3f} | {row.max_error_pp:.3f} |")
    lines.extend([
        "", "## What the action data added", "",
        "The primary comparison uses Random Walk at three tests ahead. Each row compares the same held-out "
        "windows. A negative difference favors the action-aware model. Intervals resample whole batteries.", "",
        "| Baseline | Action-aware MAE change (pp) | 95% interval (pp) | Batteries with lower error |",
        "| --- | ---: | ---: | ---: |",
    ])
    for row in paired.loc[paired.cohort.eq("RW") & paired.horizon_tests.eq(3)].itertuples(index=False):
        lines.append(f"| {MODEL_LABELS[row.baseline]} | {row.mean_mae_change_pp:+.3f} | "
                     f"{row.ci95_low_pp:+.3f} to {row.ci95_high_pp:+.3f} | "
                     f"{row.action_better_batteries}/{row.batteries} |")
    actions = ", ".join(f"{amps} A for {seconds // 60} min" for amps, seconds in common_actions)
    coverage = actions if actions else "No pair met the overlap rule"
    operating = cycles.loc[cycles.cohort.eq("RW") & cycles.status.eq("valid_operating")]
    audited_durations = operating.verified_first_duration_s.notna().sum()
    excluded_intervals = len(intervals) - len(valid)
    lines.extend([
        "",
        f"**Comparable choices across folds** · {coverage}. Each pair had records on at least "
        f"{MIN_ACTION_BATTERIES} training batteries in both reference SOH bands from 80% to under 90% "
        "and from 90% upward. The gate requires these pairs to cover at least two different current levels.",
        "",
        f"Verified an initial current–duration pair in {audited_durations}/{len(operating)} Random Walk "
        f"operating cycles. The interval audit excluded {excluded_intervals} intervals across both cohorts. "
        "The forecast treats the first plateaus as the prescribed schedule. Remaining discharge and "
        "cutoff-dependent timing are observational and enter the retrospective model only.",
        "",
        "The R3 comparison stays separate because its verified intervals repeat the same five rates and use a "
        "different diagnostic charging current. The cohort tests forecast accuracy under that schedule.", "",
        "[Original XJTU experimental protocols](https://wang-fujin.github.io/) · "
        "[XJTU battery dataset](https://zenodo.org/records/10963339)", "",
    ])
    return "\n".join(lines)


def plot_results(cycles, metrics, path):
    import matplotlib.pyplot as plt
    figure, axes = plt.subplots(1, 2, figsize=(11, 4.25), layout="constrained")
    subset = cycles.loc[cycles.cohort.eq("RW") & cycles.status.eq("valid_rpt")]
    for cell, rows in subset.groupby("cell_id", sort=True):
        axes[0].plot(rows.cycle, rows.diagnostic_soh_pct, marker=".", linewidth=1.2, label=cell.rsplit("-", 1)[-1])
    axes[0].set(title="Random Walk reference tests", xlabel="Recorded cycle", ylabel="SOH (%)")
    axes[0].legend(ncol=4, fontsize=8, title="Battery")
    groups = metrics.loc[metrics.cohort.eq("RW") & metrics.model.ne("realized")]
    labels = [MODEL_LABELS[key] for key in MODEL_LABELS if key != "realized"]
    positions = np.arange(len(labels))
    for position, count in enumerate(HORIZONS):
        values = (groups.loc[groups.horizon_tests.eq(count)].set_index("model")
                  .reindex(list(MODEL_LABELS)[:-1]).mae_pp)
        axes[1].barh(positions + (position - .5) * .31, values, height=.31, label=f"{count} test" + ("s" if count > 1 else ""),
                     color=("#3d739c", "#22846f")[position])
    axes[1].set_yticks(positions, labels)
    axes[1].invert_yaxis()
    axes[1].set(title="Random Walk held-out forecasts", xlabel="Mean absolute error (SOH percentage points)")
    axes[1].legend(fontsize=8)
    for ax in axes:
        ax.grid(axis="x", alpha=.2)
    figure.savefig(path, dpi=145)
    plt.close(figure)


output.mkdir(parents=True, exist_ok=True)
tables = {
    "cycle_audit": cycles, "cell_audit": raw_checks, "reference_intervals": intervals,
    "held_out_predictions": predictions, "model_selection": model_selection, "forecast_metrics": metrics,
    "paired_error_changes": paired, "observed_first_currents": observed_commands,
    "action_overlap": support, "ppo_gate": gate,
}
for name, table in tables.items():
    table.to_csv(output / f"{name}.csv", index=False)
config = dict(source="BatteryLife XJTU", xjtu_source_doi="10.5281/zenodo.10963339", nominal_ah=2.0,
              cohorts=COHORTS, horizon_rpt_tests=HORIZONS, ridge_alphas=ALPHAS,
              bootstrap_draws=BOOTSTRAP_DRAWS, seed=SEED, health_bands=HEALTH_BANDS,
              min_batteries_per_action_and_band=MIN_ACTION_BATTERIES,
              min_common_actions=MIN_COMMON_ACTIONS, common_initial_current_duration_pairs=common_actions)
(output / "run_config.json").write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
(output / "summary.md").write_text(build_summary(metrics, paired, gate, cycles, intervals, support, common_actions),
                                   encoding="utf-8")
plot_results(cycles, metrics, output / "rpt_forecasts.png")
files = [*(f"{name}.csv" for name in tables), "run_config.json", "summary.md", "rpt_forecasts.png"]
archive_path = output.with_suffix(".zip")
with ZipFile(archive_path, "w", ZIP_DEFLATED) as bundle:
    for name in files:
        bundle.write(output / name, arcname=f"{output.name}/{name}")
print(f"Report: {output / 'summary.md'}")
print(f"Download: {archive_path}")

Report: /Users/sakibsmacbookm5/Documents/Python work/research-project/SICIP Final Project/reports/lifeguard_xjtu_discharge_soh_results/summary.md
Download: /Users/sakibsmacbookm5/Documents/Python work/research-project/SICIP Final Project/reports/lifeguard_xjtu_discharge_soh_results.zip
